In [ ]:
# Imports and LLM setup

import pandas as pd
import json
import re

from langchain_ollama import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate

# Local LLM
model = OllamaLLM(
    model="deepseek-r1:8b"
)

In [ ]:
# Trade Hostility Index Rubric

trade_rubric = """
TRADE HOSTILITY INDEX

Definition:
The Trade Hostility Index measures the level of hostility, tension, confrontation, or escalation in trade relations between the United States and another country or group of countries, as expressed in the tweet.

To assess the Trade Hostility Index, consider three dimensions:

1. SEVERITY

Very Low:
Little or no hostility. The tweet may mention trade, negotiations, or trade relations without meaningful confrontation.

Low:
Mild criticism, dissatisfaction, or tension, but no clear escalation.

Moderate:
Clear trade disagreement, pressure, criticism, or threat with meaningful hostility.

Severe:
Strong confrontation or significant escalation in trade relations, including serious threats, restrictive policy action, or major economic grievances.

Very Severe:
Major and highly consequential escalation, such as an extreme threat, major restriction, termination of an important trade relationship, or other action with potentially large economic consequences.

2. ROLE

Direct:
Trump or the US is actively driving the trade development, for example by announcing, threatening, demanding, changing, or implementing a trade action.

Indirect:
Trump is mainly commenting on, describing, reacting to, or reporting a trade issue without directly initiating a new action.

3. STATUS

Concluded:
The relevant trade development has already occurred or ended.

Partially Concluded:
Part of the issue has been resolved or completed, but the broader trade issue remains unresolved.

Ongoing:
The trade dispute, negotiation, pressure, or policy action is currently active.

Imminent:
A new trade action or escalation is expected or threatened in the near future.

FINAL SCORE

SEVERITY determines the main score range:

0: No relevant trade-hostility signal.
1-19 — Very Low
20-39 — Low
40-59 — Moderate
60-79 — Severe
80-100 — Very Severe

SCORING LOGIC

ROLE and STATUS then determine where within that range the final score should fall.

For otherwise similar tweets:

- Very Severe > Severe > Moderate > Low > Very Low
- Direct > Indirect
- Imminent or Ongoing > Partially Concluded > Concluded

Therefore:
- stronger SEVERITY gives a higher base score;
- a Direct ROLE moves the score upward relative to an Indirect ROLE;
- an Ongoing or Imminent issue moves the score upward relative to a Concluded issue.

For example, a Very Severe, Direct, Imminent tweet should normally fall toward the upper end of the 80-100 band, while a Very Severe, Indirect, Concluded tweet would normally fall toward the lower end of that band.


First assess whether the tweet genuinely contains a trade-hostility signal. 
If it does, assess its SEVERITY, ROLE, and STATUS, and finally use these three dimensions together to choose the final score. If the tweet does not contain a trade-hostility signal, assign a score of 0.
"""

In [ ]:
# Human Worked Example: Trade Hostility Index Scoring

trade_example = """
HUMAN WORKED EXAMPLE - TRADE HOSTILITY

The following example demonstrates how I would apply the Trade Hostility rubric as a human annotator.

Tweet:
"Our Steel and Aluminum industries (and many others) have been decimated by decades of unfair trade and bad policy with countries from around the world. 
We must not let our country, companies and workers be taken advantage of any longer. We want free, fair and SMART TRADE!" 

My assessment:

RELEVANCE: I first determine whether the tweet contains a genuine trade-hostility signal.

I consider this tweet relevant because Trump describes the existing international trade relationship as unfair and harmful to US industries, companies, and workers.
He expresses clear dissatisfaction with how the US is being treated and states that this situation should not continue.

I therefore proceed to assess SEVERITY, ROLE and STATUS.


SEVERITY: Severe

I classify the SEVERITY as Severe because the tweet expresses strong dissatisfaction with existing trade relations and describes their economic consequences as substantial.
Trump presents US industries and workers as having suffered significant harm over a long period and makes clear that he considers the current situation unacceptable.

However, I do not classify the SEVERITY as Very Severe. Although the language is strong, no specific foreign country or trading partner is targeted and no concrete major trade action, restriction, or policy response is announced.
The tweet signals a serious trade conflict, but not yet a speficic extreme escalation.


ROLE: Direct

I classify the ROLE as Direct because Trump is not merely reporting or commenting on a trade issue. He takes a clear position that the existing situation should no longer continue and expresses an intention for the US to seek different treatment.

Although he does not specify exactly what policy action will follow, he is directly applying pressure for a change in US trade relations.


STATUS: Ongoing

I classify the STATUS as Ongoing because the problem described in the tweet is presented as unresolved and continuing.

Trump describes the perceived harm as having occurred over a long period and states that the US should no longer accept the existing situation.
This indicates that the trade issue remains active.

I do not classify the STATUS as Imminent because no specific forthcoming action, deadline, restriction, or policy measure is announced.


FINAL SCORE: 76

Severe SEVERITY places the tweet in the 60-79 range.

The Direct ROLE and Ongoing STATUS move the score toward the upper end of that range.
The language is strong and the perceived economic consequences are substantial, so
I assign a score of 76.

I do not move the score into the Very Severe range because there is no specific
target and no concrete major action or imminent escalation described.

Final Trade Hostility Index = 76.
"""




In [ ]:
# Sanction Threat Index Rubric
sanctions_rubric = """
SANCTIONS THREAT INDEX

Definition:
The Sanctions Threat Index measures the level of threat, pressure, confrontation, or escalation associated with the use or potential use of economic sanctions by the United States against another country or foreign actor, as expressed in the tweet.

To assess the Sanctions Threat Index, consider three dimensions:

1. SEVERITY

Very Low:
Little or no sanctions threat. The tweet may refer to a geopolitical dispute or past confrontation without signalling meaningful sanctions pressure.

Low:
A weak sanctions-related signal is present, such as discussion of sanctions or economic pressure without a clear threat of escalation or major policy action.

Moderate:
Clear sanctions pressure or a meaningful threat of economic punishment is present, but the scale, scope, or consequences remain limited or uncertain.

Severe:
Strong sanctions pressure or significant escalation is present, including clear threats, continuation or strengthening of sanctions, or substantial economic pressure against a foreign target.

Very Severe:
Major and highly consequential sanctions escalation, such as the announcement, imposition, or strong threat of substantial new sanctions or other major economic punishment with potentially large consequences.

2. ROLE

Direct:
Trump or the US is actively driving the sanctions development, for example by announcing, imposing, threatening, extending, strengthening, removing, or otherwise making a decision about sanctions or economic punishment.

Indirect:
Trump is mainly commenting on, describing, reacting to, or reporting sanctions or economic pressure without directly announcing or initiating a sanctions action.

3. STATUS

Concluded:
The relevant sanctions action or event has already occurred or ended and the tweet mainly refers to it retrospectively.

Partially Concluded:
Some sanctions action has already occurred, but the broader dispute or sanctions policy remains unresolved or active.

Ongoing:
The sanctions, economic pressure, dispute, or policy action is currently active.

Imminent:
A new sanctions action, escalation, or other form of economic punishment is expected, announced, or threatened in the near future.

FINAL SCORE

SEVERITY determines the main score range:

0 — No relevant sanctions-threat signal
1-19 — Very Low
20-39 — Low
40-59 — Moderate
60-79 — Severe
80-100 — Very Severe

SCORING LOGIC

ROLE and STATUS then help determine where within that range the final score should fall.

For otherwise similar tweets:

- Very Severe > Severe > Moderate > Low > Very Low
- Direct > Indirect
- Imminent or Ongoing > Partially Concluded > Concluded

Therefore:
- stronger SEVERITY gives a higher score range;
- a Direct ROLE moves the score upward relative to an Indirect ROLE;
- an Ongoing or Imminent issue moves the score upward relative to a Partially Concluded or Concluded issue.

For example, a Very Severe, Direct, Imminent sanctions threat should normally fall toward the upper end of the 80-100 range, while a Very Severe, Indirect, Concluded sanctions event should normally fall toward the lower end of that range.

First assess whether the tweet genuinely contains a sanctions-threat signal.
If it does, assess its SEVERITY, ROLE, and STATUS and use these dimensions together to choose the final score. If it does not, assign a score of 0.
"""

In [ ]:
# Human Worked Example: Sanctions Threat Index Scoring

sanctions_example = """
HUMAN WORKED EXAMPLE — SANCTIONS THREAT 

The following example demonstrates how I, as a human annotator, apply the Sanctions Threat Index rubric to a tweet.

Tweet:
"It was announced today by the U.S. Treasury that additional large scale Sanctions would be added to those already existing Sanctions on North Korea.
I have today ordered the withdrawal of those additional Sanctions!"

My assessment:

RELEVANCE:

I first determine whether the tweet contains a genuine sanctions-threat signal.

I consider this tweet relevant because it describes an existing sanctions regime and a proposed additional large-scale sanctions action. 
The tweet therefore contains meaningful information about US sanctions policy.

I then assess its SEVERITY, ROLE, and STATUS.


SEVERITY: Moderate

I classify the SEVERITY as Moderate.

The proposed additional sanctions are described as large scale, which would represent a meaningful increase in economic pressure if implemented.

However, the tweet also states that Trump ordered those additional sanctions to be withdrawn. 
The new escalation is therefore being reversed rather than carried forward.

Existing sanctions remain relevant, so the sanctions signal is not absent, but the withdrawal substantially reduces the immediate threat represented by this particular tweet.


ROLE: Direct

I classify the ROLE as Direct because Trump is directly involved in the sanctions decision.

He explicitly states that he ordered the withdrawal of the additional sanctions.
The policy outcome described in the tweet is therefore directly connected to his own action rather than being something he is simply observing or reporting.


STATUS: Partially Concluded

I classify the STATUS as Partially Concluded.

The proposed additional sanctions have been withdrawn, meaning that this particular escalation has been stopped.

However, the tweet states that sanctions already exist. 
The wider sanctions policy has therefore not ended, so I do not classify the overall issue as fully Concluded.


FINAL SCORE: 47

Moderate SEVERITY places the tweet in the 40-59 range.

The ROLE is Direct, which supports a meaningful score within that range because Trump is directly making a sanctions-policy decision.

However, the additional sanctions are being withdrawn rather than imposed, and the attempted escalation has therefore been reduced. 
The Partially Concluded STATUS keeps the score away from the upper end of the Moderate range.

For these reasons, I assign a score of 47.

Final Sanctions Threat Index = 47.

IMPORTANT:
The specific country or actor mentioned in this example is not itself an indicator of sanctions threat. 
Apply the same reasoning process to any tweet, regardless of which country, actor, or geopolitical situation it concerns.
The score must be based on the sanctions signal described in the tweet.
"""

In [ ]:
# Federal Reserve Pressure Index Rubric

fed_rubric = """
FEDERAL RESERVE PRESSURE INDEX

Definition:
The Federal Reserve Pressure Index measures the level of pressure, criticism, confrontation, or attempted influence exerted by Donald Trump on the Federal Reserve or its monetary-policy decisions, as expressed in the tweet.

The index measures Trump's pressure on the Federal Reserve, rather than the economic effects of Federal Reserve policy itself.

To assess the Federal Reserve Pressure Index, consider three dimensions:

1. SEVERITY

Very Low:
Little or no pressure. The tweet may discuss monetary policy or the Federal Reserve without meaningful criticism, demand, or confrontation.

Low:
Mild criticism, disagreement, or dissatisfaction with Federal Reserve policy, but without strong pressure or a clear demand for change.

Moderate:
Clear criticism or pressure regarding Federal Reserve policy, decisions, or performance, including a meaningful demand or preference for policy change.

Severe:
Strong and direct pressure on the Federal Reserve, including forceful criticism, repeated blame, strong demands for policy change, or significant attacks on its decisions or performance.

Very Severe:
Extreme and highly confrontational pressure on the Federal Reserve, including particularly aggressive attacks, very strong demands for major policy change,
or other statements representing an unusually high level of attempted influence over Federal Reserve decision-making.

2. ROLE

Direct:
Trump himself directly criticises, pressures, attacks, blames, or makes demands of the Federal Reserve or its decision-makers.

Indirect:
The tweet discusses criticism, pressure, or opinions concerning Federal Reserve policy without Trump himself directly applying that pressure, for example by
reporting or referring to another person's view.

3. STATUS

Concluded:
The relevant Federal Reserve decision or event has already occurred, and the
tweet mainly comments on it retrospectively.

Partially Concluded:
A Federal Reserve decision or policy action has already occurred, but Trump continues to express pressure or dissatisfaction and the broader issue remains unresolved.

Ongoing:
Trump's pressure or criticism concerns a Federal Reserve policy issue that remains active or unresolved.

Imminent:
Trump is applying pressure regarding a Federal Reserve decision or policy action that is expected in the near future.

FINAL SCORE

SEVERITY determines the main score range:

0 — No relevant Federal Reserve pressure signal
1-19 — Very Low
20-39 — Low
40-59 — Moderate
60-79 — Severe
80-100 — Very Severe

SCORING LOGIC

ROLE and STATUS then help determine where within that range the final score
should fall.

For otherwise similar tweets:

- Very Severe > Severe > Moderate > Low > Very Low
- Direct > Indirect
- Imminent or Ongoing > Partially Concluded > Concluded

Therefore:
- stronger SEVERITY gives a higher score range;
- a Direct ROLE moves the score upward relative to an Indirect ROLE;
- an Ongoing or Imminent issue moves the score upward relative to a Partially Concluded or Concluded issue.

For example, Very Severe, Direct and Ongoing pressure should normally fall toward the upper end of the 80-100 range, while a similarly severe but Indirect
and Concluded case should normally fall toward the lower end of that range.

First assess whether the tweet genuinely contains a Federal Reserve pressure signal. 
If it does, assess its SEVERITY, ROLE, and STATUS and use these dimensions together to choose the final score. 
If it does not, assign a score of 0.
"""

In [ ]:
# Human Worked Example: Federal Reserve Pressure Index Scoring

fed_example = """
HUMAN WORKED EXAMPLE — FEDERAL RESERVE PRESSURE 

The following example demonstrates how I, as a human annotator, apply the Federal Reserve Pressure Index rubric to a tweet.

Tweet:
"It is far more costly for the Federal Reserve to cut deeper if the economy actually does, in the future, turn down! 
Very inexpensive, in fact productive, to move now. The Fed raised &, tightened far too much &, too fast. 
In other words, they missed it (Big!). Don't miss it again!"

My assessment:

RELEVANCE:

I first determine whether the tweet contains a genuine Federal Reserve pressure signal.

I consider this tweet relevant because Trump is directly criticising the Federal Reserve's previous monetary-policy decisions while also arguing that it should
change its current policy.

He is therefore not simply discussing monetary policy or its economic effects.
He is expressing dissatisfaction with the Fed's decisions and applying pressure for it to act differently.

I then assess SEVERITY, ROLE, and STATUS.


SEVERITY: Severe

I classify the SEVERITY as Severe because the pressure is strong and explicit.

Trump argues that the Fed tightened monetary policy too much and too quickly, states that it made a significant mistake, and argues that it should act now rather than waiting for a future economic downturn.

The criticism is therefore combined with a clear preference for how the Fed should conduct monetary policy.

However, I do not classify the SEVERITY as Very Severe. 
Although the criticism and pressure are strong, the tweet remains primarily a forceful criticism of policy and a demand for different monetary-policy action. 
It does not contain the more extreme form of confrontation or attempted influence that I would require for the Very Severe category.


ROLE: Direct

I classify the ROLE as Direct because the pressure comes directly from Trump.

He personally criticises the Fed's previous decisions and explicitly tells it not to repeat what he considers to have been a mistake.

The pressure is therefore being applied by Trump himself rather than being a reference to somebody else's criticism or opinion.


STATUS: Ongoing

I classify the STATUS as Ongoing.

Although Trump refers to previous tightening decisions that have already occurred, his criticism is being used to argue for a change in current and
future monetary policy.

The issue is therefore still active: he is applying pressure in the present for the Federal Reserve to act differently.


FINAL SCORE: 79

Severe SEVERITY places the tweet in the 60-79 range.

The ROLE is Direct and the pressure is Ongoing, which moves the score toward the very top of that range. 
The criticism is strong, the Fed is explicitly blamed for previous mistakes, and Trump directly argues for a different policy response.

For these reasons, I assign a score of 79, the upper end of the Severe range.

I do not move the score into the 80-100 Very Severe range because, despite the strong and direct pressure, the tweet does not contain the more extreme level
of confrontation or attempted influence required to move into the next category.

Final Federal Reserve Pressure Index = 79.
"""

In [ ]:
# Final Prompt

system_instructions = f"""
You are a geopolitical risk analyst.

Your task is to analyse individual Donald Trump tweets and independently assign three scores:

1. Trade Hostility Index
2. Sanctions Threat Index
3. Federal Reserve Pressure Index

Use the following rubrics and human worked examples as your scoring methodology.

{trade_rubric}

{trade_example}

{sanctions_rubric}

{sanctions_example}

{fed_rubric}

{fed_example}

GENERAL INSTRUCTIONS

Assess all three indices independently for every tweet.

A tweet may:
- be relevant to only one index;
- be relevant to more than one index;
- or contain no relevant signal for any index.

Do not assume that relevance to one index implies relevance to another.

For each index, follow the same process demonstrated in the worked examples:

1. Determine whether the tweet is relevant.
2. Assess SEVERITY.
3. Assess ROLE.
4. Assess STATUS.
5. Use these assessments together to choose the final integer score.

Return a brief explanation showing why you assigned the three final scores.

Return ONLY a valid JSON object in exactly the following structure:

{{
    "trade_score": <integer from 0 to 100>,
    "sanction_score": <integer from 0 to 100>,
    "fed_pressure_score": <integer from 0 to 100>,
    "thinking": "<brief explanation of the reasoning behind the three scores>"
}}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_instructions),
    ("human", """
Analyse the following tweet:

Tweet ID: {tweet_id}
Date: {date}
Tweet: {tweet}
""")
])

chain = prompt | model

In [ ]:
# Test the model on one tweet with detailed reasoning

test_prompt = ChatPromptTemplate.from_messages([
    ("system", system_instructions),
    ("human", """
Analyse the following tweet:

Tweet ID: {tweet_id}
Date: {date}
Tweet: {tweet}

For this TEST ONLY, give a detailed explanation similar in structure to the human worked examples.

For each of the three indices:

1. Explain whether the tweet is relevant.
2. If it is not relevant, assign a score of 0 and explain briefly why.
3. If it is relevant:
    - assess Severity and explain why;
    - assess Role and explain why;
    - assess Status and explain why;
    - explain how those dimensions lead to the final numerical score.
4. Give the final score.

Return the result as valid JSON in exactly this structure:

{{
    "trade": {{
        "relevance": "<explanation>",
        "severity": "<category and explanation>",
        "role": "<category and explanation>",
        "status": "<category and explanation>",
        "score_reasoning": "<how the score was chosen>",
        "score": <integer from 0 to 100>
    }},
    "sanctions": {{
        "relevance": "<explanation>",
        "severity": "<category and explanation>",
        "role": "<category and explanation>",
        "status": "<category and explanation>",
        "score_reasoning": "<how the score was chosen>",
        "score": <integer from 0 to 100>
    }},
    "fed_pressure": {{
        "relevance": "<explanation>",
        "severity": "<category and explanation>",
        "role": "<category and explanation>",
        "status": "<category and explanation>",
        "score_reasoning": "<how the score was chosen>",
        "score": <integer from 0 to 100>
    }}
}}
""")
])

test_chain = test_prompt | model

(For the following tweet, my own scores read: 
Trade Hostility = 58
Sanctions Threat = 45
Fed Pressure = 0)

In [ ]:
test_tweet = """
Many people talking, with much agreement, on my Iran speech today. Participants in the deal are making lots of money on trade with Iran!
"""


test_response = test_chain.invoke({
    "tweet_id": "919005534883328e3",
    "date": "2017-10-14 01:02:48",
    "tweet": test_tweet
})

print(test_response)

In [ ]:
# Annotate full cleaned tweet dataset

INPUT_FILE = "../data/processed/cleaned_tweets.csv"
OUTPUT_FILE = "../data/processed/LLM_annotations.csv"

# Number of tweets to process.
# Keep at 10 for the first test.
# Change to None to process the full dataset.
MAX_ROWS = 10


# Load cleaned + merged tweets

df = pd.read_csv(INPUT_FILE)

required_columns = ["Tweet ID", "Date", "Tweet"]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}. "
        f"Available columns: {list(df.columns)}"
    )


# Use a small sample for testing, or the full dataset
if MAX_ROWS is not None:
    df_to_score = df.head(MAX_ROWS).copy()
else:
    df_to_score = df.copy()


# Output CSV

output_columns = [
    "Tweet ID",
    "Date",
    "Tweet",
    "Trade Hostility Index",
    "Sanctions Threat Index",
    "Fed Pressure Index",
    "Brief Explanation"
]

# Creates a new CSV containing the column headers
pd.DataFrame(columns=output_columns).to_csv(
    OUTPUT_FILE,
    index=False
)


# Iterative process for scoring each tweet

for position, (_, row) in enumerate(df_to_score.iterrows(), start=1):

    print(f"Processing tweet {position}/{len(df_to_score)}...")

    try:

        # Send this tweet to the LLM
        response = chain.invoke({
            "tweet_id": str(row["Tweet ID"]),
            "date": str(row["Date"]),
            "tweet": str(row["Tweet"])
        })


        # Remove DeepSeek <think>...</think> text if returned
        cleaned_response = re.sub(
            r"<think>.*?</think>",
            "",
            response,
            flags=re.DOTALL
        ).strip()


        # Find the JSON object returned by the LLM
        json_match = re.search(
            r"\{.*\}",
            cleaned_response,
            flags=re.DOTALL
        )

        if json_match is None:
            raise ValueError("No JSON object found in model response.")


        # Convert JSON response into Python dictionary
        scores = json.loads(json_match.group())


        # Create the output row
        result = {
            "Tweet ID": row["Tweet ID"],
            "Date": row["Date"],
            "Tweet": row["Tweet"],
            "Trade Hostility Index": scores["trade_score"],
            "Sanctions Threat Index": scores["sanction_score"],
            "Fed Pressure Index": scores["fed_pressure_score"],
            "Brief Explanation": scores["thinking"]
        }


    except Exception as e:

        print(
            f"Error processing Tweet ID "
            f"{row['Tweet ID']}: {e}"
        )

        # Keep the tweet in the output even if processing fails
        result = {
            "Tweet ID": row["Tweet ID"],
            "Date": row["Date"],
            "Tweet": row["Tweet"],
            "Trade Hostility Index": None,
            "Sanctions Threat Index": None,
            "Fed Pressure Index": None,
            "Brief Explanation": f"ERROR: {e}"
        }


    # Append this tweet to output CSV

    pd.DataFrame([result]).to_csv(
        OUTPUT_FILE,
        mode="a",
        header=False,
        index=False
    )



print("\nFinished.")
print(f"Tweets processed: {len(df_to_score)}")
print(f"Results saved to: {OUTPUT_FILE}")